In [ ]:
# Paths: set DATA_DIR / OUTPUT_DIR as environment variables, or edit the defaults below.
import os
DATA_DIR = os.environ.get("DATA_DIR", "../data")


In [1]:
pass  # Colab only: from google.colab import drive
pass  # Colab only: drive.mount('/content/drive')
PROJECT_DIR = DATA_DIR + "/"

Mounted at /content/drive


In [2]:
!pip install osmnx geopandas shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 2.1 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import osmnx as ox
from shapely.geometry import Point
import geopandas as gpd

SAVE_DIR = DATA_DIR + "/datasets/"

df = pd.read_csv(SAVE_DIR + "collisions_features.csv",
                 dtype={'collision_index': str},
                 low_memory=False)

print(f"Loaded {len(df):,} accidents")


Loaded 503,407 accidents


In [4]:
# 1. Converting accidents to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
    crs='EPSG:4326'
)

In [5]:
# 2. Downloading road network for a pilot area first (we'll scale up after)
# Using Greater Manchester as pilot area (decent mix of urban/rural road networks)
G = ox.graph_from_place("Greater Manchester, UK", network_type='drive')
edges = ox.graph_to_gdfs(G, nodes=False)
edges = edges.reset_index()
edges = edges.to_crs('EPSG:4326')

print(f"Road segments in Greater Manchester: {len(edges):,}")
print(f"Edge columns: {list(edges.columns[:10])}")  # first 10

Road segments in Greater Manchester: 234,282
Edge columns: ['u', 'v', 'key', 'osmid', 'highway', 'lanes', 'maxspeed', 'oneway', 'ref', 'reversed']


In [6]:
# 3. Cliping accidents to Greater Manchester bounding box
bounds = edges.total_bounds
mask = (
    (gdf['longitude'] >= bounds[0]) & (gdf['longitude'] <= bounds[2]) &
    (gdf['latitude']  >= bounds[1]) & (gdf['latitude']  <= bounds[3])
)
gdf_pilot = gdf[mask].copy()
print(f"\nAccidents within Greater Manchester bounds: {len(gdf_pilot):,}")


Accidents within Greater Manchester bounds: 16,949


In [7]:
# 4. Snaping each accident to its nearest road segment
print("\nSnapping accidents to nearest road segments...")
edges_proj = edges.to_crs('EPSG:27700')   # British National Grid (metres)
gdf_proj   = gdf_pilot.to_crs('EPSG:27700')

# For each accident point, find the nearest edge
from shapely.ops import nearest_points

# Creating a unified edge geometry for spatial index
edges_proj['edge_id'] = edges_proj.index
edge_union = edges_proj.geometry.unary_union

nearest_edge_ids = []
for point in gdf_proj.geometry:
    nearest_geom = nearest_points(point, edge_union)[1]
    # Finding which edge this point belongs to
    distances = edges_proj.geometry.distance(point)
    nearest_edge_ids.append(distances.idxmin())

gdf_pilot['edge_id'] = nearest_edge_ids
print(f"Snapping complete. Unique segments with accidents: {gdf_pilot['edge_id'].nunique():,}")


Snapping accidents to nearest road segments...


/tmp/ipython-input-254/2419376512.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  edge_union = edges_proj.geometry.unary_union


Snapping complete. Unique segments with accidents: 10,895


In [8]:
# 5. Building segment-level features
seg = gdf_pilot.groupby('edge_id').agg(
    total_accidents      = ('collision_index', 'count'),
    serious_accidents    = ('is_serious', 'sum'),
    fatal_accidents      = ('collision_severity', lambda x: (x==1).sum()),
    avg_speed_limit      = ('speed_limit', 'mean'),
    pct_dark             = ('is_dark', 'mean'),
    pct_wet              = ('is_wet_road', 'mean'),
    pct_rush_hour        = ('is_rush_hour', 'mean'),
    pct_bad_weather      = ('is_bad_weather', 'mean'),
    pct_at_junction      = ('at_junction', 'mean'),
).reset_index()

seg['serious_rate']  = seg['serious_accidents'] / seg['total_accidents']
seg['accidents_per_year'] = seg['total_accidents'] / 5  # 5 years of data

In [9]:
# Merging road properties from OSM edges
edge_props = edges_proj[['edge_id','length','highway']].copy()
seg = seg.merge(edge_props, on='edge_id', how='left')

print(f"\nSegment feature matrix shape: {seg.shape}")
print(f"\nTop 10 highest accident segments:")
print(seg.nlargest(10, 'total_accidents')[
    ['edge_id','total_accidents','serious_accidents',
     'serious_rate','avg_speed_limit','length']
])


Segment feature matrix shape: (10895, 14)

Top 10 highest accident segments:
       edge_id  total_accidents  serious_accidents  serious_rate  \
4680     70103              688                122      0.177326   
7771    143899              301                 59      0.196013   
4337     64713              272                 76      0.279412   
4679     70102              249                 33      0.132530   
4766     72999              179                 38      0.212291   
3679     55719              168                 36      0.214286   
1633     17017              152                 30      0.197368   
3865     57773              131                 34      0.259542   
10076   202798               95                 14      0.147368   
10352   213432               83                 24      0.289157   

       avg_speed_limit      length  
4680         35.116279  173.126732  
7771         37.674419   72.815328  
4337         35.588235  114.282623  
4679         30.923695   

In [10]:
# Save
seg.to_csv(SAVE_DIR + "segment_features_manchester.csv", index=False)
print("\n Saved segment_features_manchester.csv")


 Saved segment_features_manchester.csv
